<a href="https://colab.research.google.com/github/jcalmeida-ds/entrevistador-por-voz-multilingue/blob/main/entrevistador_por_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Entrevistador por Voz Multilíngue
### Whisper (STT) + ChatGPT + gTTS (TTS)

Simulador de entrevista de emprego por voz. Você responde falando, o sistema transcreve (Whisper), o entrevistador (ChatGPT) responde no mesmo idioma e a resposta é falada (gTTS). Ao final, gera um **feedback estruturado** da sua performance.

**Fluxo**

```
Microfone → Whisper (STT) → texto + idioma → ChatGPT (entrevistador) → gTTS (TTS) → áudio
                                                    ↓
                                          histórico → feedback final
```

**Modos disponíveis:** fit cultural, entrevista técnica (IA / prompt engineering) e entrevista em inglês.

## 1. Instalação e configuração

A chave da OpenAI fica nos **Secrets** do Colab (ícone 🔑 na barra lateral, nome `OPENAI_API_KEY`). Nunca escreva a chave no código.

In [27]:
!pip install -q openai gTTS

In [29]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODELO_CHAT = "gpt-4o-mini"
MODELO_STT = "whisper-1"
print("Cliente OpenAI configurado.")

Cliente OpenAI configurado.


## 2. Gravação de áudio no Colab

O Colab não acessa o microfone diretamente, então usamos JavaScript no navegador. A função `gravar()` grava até você clicar em **Parar**.

In [30]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode

RECORD_JS = '''
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  let chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    const blob = new Blob(chunks);
    const text = await b2text(blob);
    resolve(text);
  };
  recorder.stop();
  stream.getTracks().forEach(t => t.stop());
});
'''

def gravar(segundos=30, arquivo="entrada.webm"):
    display(Javascript(RECORD_JS))
    print(f"🔴 Gravando por até {segundos}s... fale agora.")
    dados = eval_js(f"record({segundos * 1000})")
    binario = b64decode(dados.split(",")[1])
    with open(arquivo, "wb") as f:
        f.write(binario)
    print("⏹️ Gravação finalizada.")
    return arquivo

Alternativa, caso a gravação falhe no seu navegador: envie um arquivo de áudio pronto.

In [31]:
from google.colab import files

def enviar_audio():
    up = files.upload()
    return list(up.keys())[0]

## 3. Speech-to-Text com Whisper

O Whisper devolve o texto e o **idioma detectado** (como nome, ex.: `portuguese`).

In [32]:
def transcrever(caminho):
    with open(caminho, "rb") as f:
        r = client.audio.transcriptions.create(
            model=MODELO_STT,
            file=f,
            response_format="verbose_json",
        )
    return r.text.strip(), r.language

## 4. Modos de entrevista (prompts de sistema)

Trocar o modo é só trocar o prompt. Aqui está o coração do projeto em termos de **engenharia de prompt**.

In [33]:
BASE = '''
Você é um entrevistador profissional realizando uma simulação de entrevista de emprego.
Regras:
- Responda SEMPRE no mesmo idioma em que o candidato falou.
- Faça UMA pergunta por vez e espere a resposta.
- Suas falas serão lidas em voz alta: use frases curtas, sem markdown, listas, emojis ou símbolos.
- Se a resposta do candidato for vaga, peça um exemplo concreto (método STAR: situação, tarefa, ação, resultado).
- Reaja brevemente à resposta (uma frase) antes de fazer a próxima pergunta.
- Não dê feedback detalhado durante a entrevista; ele será pedido ao final.
'''

MODOS = {
    "fit_cultural": BASE + '''
Foco: fit cultural. Explore colaboração, aprendizado rápido, lidar com erros, feedback, ownership,
trabalho sob ambiguidade e motivação para a empresa e a vaga.
''',
    "tecnica_ia": BASE + '''
Foco: vaga de engenharia de prompt / IA generativa. Explore técnicas de prompting, avaliação de
respostas de LLMs, alucinações, uso de APIs, RAG em nível conceitual, Python e como o candidato
mede qualidade e custo de uma solução.
''',
    "ingles": BASE + '''
Foco: entrevista em inglês. Conduza em inglês. Se o candidato errar algo relevante, reformule a
pergunta de forma simples, sem corrigir durante a conversa.
''',
}

MODO_ATUAL = "fit_cultural"   # altere aqui: fit_cultural | tecnica_ia | ingles
CARGO = "Trainee em IA Generativa"   # personalize com a vaga que você quer treinar
EMPRESA = "Bradesco"                          # opcional

## 5. Conversa com histórico

In [34]:
def iniciar_historico(modo=MODO_ATUAL, cargo=CARGO, empresa=EMPRESA):
    contexto = f"\nVaga: {cargo}." + (f" Empresa: {empresa}." if empresa else "")
    return [{"role": "system", "content": MODOS[modo] + contexto}]

historico = iniciar_historico()

def chamar_chat(mensagens, temperatura=0.7):
    r = client.chat.completions.create(
        model=MODELO_CHAT, messages=mensagens, temperature=temperatura
    )
    return r.choices[0].message.content.strip()

def perguntar(texto_usuario):
    historico.append({"role": "user", "content": texto_usuario})
    resposta = chamar_chat(historico)
    historico.append({"role": "assistant", "content": resposta})
    return resposta

## 6. Text-to-Speech com gTTS

O Whisper devolve o nome do idioma e o gTTS espera código ISO, então usamos um mapeamento.

In [35]:
from gtts import gTTS
from IPython.display import Audio, display

MAPA_IDIOMAS = {
    "portuguese": "pt", "english": "en", "spanish": "es",
    "german": "de", "french": "fr", "italian": "it",
}

def falar(texto, idioma_whisper="portuguese", arquivo="resposta.mp3"):
    codigo = MAPA_IDIOMAS.get(idioma_whisper.lower(), "en")
    gTTS(texto, lang=codigo).save(arquivo)
    display(Audio(arquivo, autoplay=True))

## 7. Um turno completo da entrevista

In [36]:
def turno(segundos=45, usar_upload=False):
    caminho = enviar_audio() if usar_upload else gravar(segundos)
    texto, idioma = transcrever(caminho)
    print(f"🗣️ Você ({idioma}): {texto}\n")
    resposta = perguntar(texto)
    print(f"🎤 Entrevistador: {resposta}\n")
    falar(resposta, idioma)
    return idioma

## 8. Iniciando a entrevista

O entrevistador abre a conversa. Depois, rode a célula do **turno** quantas vezes quiser (uma por resposta).

In [38]:
historico = iniciar_historico()
abertura = perguntar("Olá, estou pronto para começar a entrevista. Pode fazer a primeira pergunta.")
print("🎤 Entrevistador:", abertura)
falar(abertura, "portuguese")   # use "english" no modo inglês

🎤 Entrevistador: Olá! Que bom que você está aqui. Começando: como você costuma colaborar com colegas em projetos?


In [41]:
# ▶️ Rode esta célula a cada resposta sua
idioma_atual = turno(segundos=45)

<IPython.core.display.Javascript object>

🔴 Gravando por até 45s... fale agora.
⏹️ Gravação finalizada.
🗣️ Você (portuguese): Bom, eu busco, através de colaborar, através de desafios práticos, né, resolvendo soluções dentro de um desafio, para trazer uma melhoria para todos. Então, através de ideias, de acordo com o desafio, eu busco colaborar com novas ideias, soluções práticas do dia-a-dia. É isso.

🎤 Entrevistador: Entendi, você busca trazer melhorias por meio de ideias. Pode me dar um exemplo concreto de uma situação em que você colaborou com sua equipe para resolver um desafio? Use o método STAR, por favor.



## 9. Feedback final

Ao terminar, peça uma avaliação estruturada. O texto do feedback é só para leitura (sem áudio), e a temperatura baixa deixa a avaliação mais consistente.

In [42]:
PROMPT_FEEDBACK = '''
A entrevista terminou. Com base em toda a conversa acima, escreva um feedback para o candidato,
no idioma da entrevista, neste formato:

1. Nota geral de 0 a 10 e uma frase de justificativa.
2. Três pontos fortes, com trechos que os sustentam.
3. Três pontos a melhorar, com sugestão prática para cada um.
4. Uso do método STAR: quais respostas tiveram situação, tarefa, ação e resultado claros.
5. Uma versão melhorada da resposta mais fraca.
6. Três perguntas para treinar na próxima rodada.
'''

def gerar_feedback():
    mensagens = historico + [{"role": "user", "content": PROMPT_FEEDBACK}]
    return chamar_chat(mensagens, temperatura=0.3)

feedback = gerar_feedback()
print(feedback)

1. Nota geral: 6. Você demonstrou boas intenções de colaboração, mas suas respostas foram um pouco vagas.

2. Pontos fortes:
   - Proatividade: Você menciona buscar melhorias e soluções práticas, o que é essencial em um ambiente colaborativo.
   - Foco em desafios: A ideia de resolver problemas através de desafios práticos mostra que você está disposto a se engajar.
   - Criatividade: A busca por novas ideias é um bom indicativo de inovação e adaptabilidade.

3. Pontos a melhorar:
   - Clareza nas respostas: Tente ser mais específico em suas experiências. Sugestão: prepare exemplos concretos antes da entrevista.
   - Uso do método STAR: Pratique estruturar suas respostas usando a situação, tarefa, ação e resultado. Sugestão: faça um exercício de storytelling.
   - Conexão com a vaga: Relacione suas experiências diretamente com a área de IA Generativa. Sugestão: estude mais sobre a área e como suas habilidades se aplicam.

4. Uso do método STAR: Sua resposta não apresentou claramente os

## 10. Salvar a sessão

In [43]:
import json, datetime

def salvar_sessao(caminho=None):
    caminho = caminho or f"sessao_{datetime.datetime.now():%Y%m%d_%H%M}.json"
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump({"modo": MODO_ATUAL, "cargo": CARGO, "historico": historico[1:]},
                  f, ensure_ascii=False, indent=2)
    print("Sessão salva em", caminho)

salvar_sessao()

Sessão salva em sessao_20261114_1504.json


## 11. Teste multilíngue

Para demonstrar o aspecto multi-idioma, rode a mesma pergunta em três idiomas e registre os resultados.

| Idioma | Pergunta de teste | Idioma detectado | Resposta no mesmo idioma? |
|---|---|---|---|
| Português | "Fale sobre você." | | |
| English | "Tell me about yourself." | | |
| Español | "Cuéntame sobre ti." | | |

## 12. Limitações e próximos passos

- **Latência:** cada turno passa por três serviços em sequência.
- **Custo:** Whisper e Chat são pagos por uso. Monitore o consumo.
- **Voz do gTTS:** robótica e sem controle de emoção; dá para trocar por outro TTS.
- **Sem detecção automática de fim de fala:** hoje a gravação tem tempo fixo.
- **Próximos passos:** interface com Gradio, métricas de fala (tempo, palavras de preenchimento), banco de perguntas por vaga e avaliação automática com rubrica.